In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

#### Reading File in DataFrame

In [0]:
df_matches = spark.read.format('csv')\
            .option('header', 'true')\
            .option('inferSchema', 'true')\
            .load('/Workspace/Users/asyedshabeeradhnan01@gmail.com/Data-science/raw/ipl_matches.csv')

In [0]:
df_eachball_info = spark.read.format('csv')\
               .option('header', 'true')\
               .option('inferSchema', 'true')\
               .load('/Workspace/Users/asyedshabeeradhnan01@gmail.com/Data-science/raw/ipl_ball_by_ball.csv')

#### Filtering player data for seasons >= 2020

In [0]:
df_period = df_eachball_info.withColumn('season',when(col('season').rlike(r"^\d{4}/\d{2}$"),year(col('date'))).otherwise(col('season')))\
              .withColumn('season', col('season').cast(IntegerType()))\
              .filter(col('season') >= 2020)

## Bowler Performance Aggregations

#### Calculations of total overs

In [0]:
df_total_overs = df_period.groupBy('bowler').agg(
   # Total Overs and Balls Bowled
   floor(round(count('over')/6)).alias('total_overs_bowled'),
   count('over').alias('total_balls_bowled'),
   sum('total_runs').alias('total_runs_given'),

   # Powerplay    
   floor(round(count(when(col('is_powerplay') == 1,col('over')))/6,1)).alias('overs_bowled_in_po'),
   sum(when(col('is_powerplay') == 1,col('total_runs'))).alias('runs_in_po'),
    
   # Middle Overs 
   floor(round(count(when(col('is_middle_overs') == 1,col('over')))/6,1))\
       .alias('overs_bowled_in_mo'),
   sum(when(col('is_middle_overs') == 1,col('total_runs'))).alias('runs_in_mo'),

   # Death Overs    
   floor(round(count(when(col('is_death_overs') == 1,col('over')))/6,1))\
       .alias('overs_bowled_in_do'),
   sum(when(col('is_death_overs') == 1,col('total_runs'))).alias('runs_in_do'),
   )

#### Calculation of wickets Metrics

In [0]:
df_wickets = df_period.groupBy('bowler').agg(
    # Wickets 
   count(when(col('is_wicket')==1,1)).alias('wickets'),
   
   # Wickets in PowerPlay    
   sum(when(col('is_powerplay')==1,col('is_wicket'))).alias('wickets_in_po'),

   # Wickets in Middle Overs    
   sum(when(col('is_middle_overs')==1,col('is_wicket'))).alias('wickets_in_mo'),

   # Wickets in Death Overs    
   sum(when(col('is_death_overs')==1,col('is_wicket'))).alias('wickets_in_do')
)

#### Joining Dataframe of overs and wickets

In [0]:
wickets_overs_df = df_total_overs.join(df_wickets, how='left', on='bowler')

In [0]:
wickets_overs_df.display()

#### Calculation of Bowler Metrics - Economy,Average,Strike Rate

In [0]:
bowling_metrics_df = wickets_overs_df\
    .withColumn('economy_rate'\
            ,round(try_divide(col('total_runs_given'),col('total_overs_bowled'))))\
    .withColumn('economy_rate_in_po'\
            ,round(try_divide(col('runs_in_po'),col('overs_bowled_in_po'))))\
    .withColumn('economy_rate_in_mo'\
            ,round(try_divide(col('runs_in_mo'),col('overs_bowled_in_mo'))))\
    .withColumn('economy_rate_in_do'\
            ,round(try_divide(col('runs_in_do'),col('overs_bowled_in_do'))))\
    .withColumn('bowling_avg'\
            ,round(try_divide(col('total_runs_given'),col('wickets'))))\
    .withColumn('avg_in_po'\
            ,round(try_divide(col('runs_in_po'),col('wickets_in_po'))))\
    .withColumn('avg_in_mo'\
            ,round(try_divide(col('runs_in_mo'),col('wickets_in_mo'))))\
    .withColumn('avg_in_do'\
            ,round(try_divide(col('runs_in_do'),col('wickets_in_do'))))\
    .withColumn('strike_rate'\
            ,round(try_divide(col('total_balls_bowled'),col('wickets'))))\
    .withColumn('strike_rate_in_po'\
            ,round(try_divide(col('overs_bowled_in_po'),col('wickets_in_po'))))\
    .withColumn('strike_rate_in_mo'\
            ,round(try_divide(col('overs_bowled_in_mo'),col('wickets_in_mo'))))\
    .withColumn('strike_rate_in_do'\
            ,round(try_divide(col('overs_bowled_in_do'),col('wickets_in_do'))))

In [0]:
bowling_metrics_df.display()

#### Wides and No-Balls Metrics

In [0]:
df_wides_noballs = df_period.groupBy('bowler').agg(
                    # Overs
                    floor(round(count('over')/6)).alias('total_overs_bowled'),

                    # Total Wides & Runs
                    count(when(col('wides') != 0,col('wides'))).alias('wides'),
                    sum(when(col('wides') != 0,col('total_runs'))).alias('runs_in_wides'),

                    # Wides in PowerPlay
                    count(when((col('is_powerplay')==1) & (col('wides')!=0),col('wides')))\
                        .alias('wides_in_po'),
                    sum(when((col('is_powerplay')==1) & (col('wides')!=0),col('total_runs'))).alias('runs_in_wides_po'),
                    
                    # Wides in Middle Overs
                    count(when((col('is_middle_overs')==1) & (col('wides')!=0),col('wides')))\
                        .alias('wides_in_mo'),
                    sum(when((col('is_middle_overs')==1) & (col('wides')!=0),col('total_runs'))).alias('runs_in_wides_mo'),

                    # Wides in Death Overs
                    count(when((col('is_death_overs')==1) & (col('wides')!=0),col('wides')))\
                        .alias('wides_in_do'),
                    sum(when((col('is_death_overs')==1) & (col('wides')!=0),col('total_runs'))).alias('runs_in_wides_do'),

                    # NoBalls
                    count(when(col('noballs') != 0,col('noballs'))).alias('noballs'),
                    sum(when(col('noballs') != 0,col('total_runs'))).alias('runs_in_noballs'),

                    # NoBalls in PowerPlay
                    count(when((col('is_powerplay')==1) & (col('noballs')!=0),col('wides')))\
                        .alias('noballs_in_po'),
                    sum(when((col('is_powerplay')==1) & (col('noballs')!=0),col('total_runs'))).alias('runs_in_noballs_po'),

                    # NoBalls in Middle Overs
                    count(when((col('is_middle_overs')==1) & (col('noballs')!=0),col('wides')))\
                        .alias('noballs_in_mo'),
                    sum(when((col('is_middle_overs')==1) & (col('noballs')!=0),col('total_runs'))).alias('runs_in_noballs_mo'),

                    # NoBalls in Death Overs
                    count(when((col('is_death_overs')==1) & (col('noballs')!=0),col('wides')))\
                        .alias('noballs_in_do'),
                    sum(when((col('is_death_overs')==1) & (col('noballs')!=0),col('total_runs'))).alias('runs_in_noballs_do')
                    )

In [0]:
df_wides_noballs.display()

#### Runs Conceeded in Boundaries Metrics

In [0]:
df_no_of_4s = df_period.filter(col('total_runs')==4)\
            .groupBy('bowler')\
            .agg(
            count(col('total_runs')).alias('no_of_4s'),
            sum(col('total_runs')).alias('runs_conceeded_in_boundaries'),

            # 4s in PowerPlay
            count(when(col('is_powerplay')==1,col('total_runs'))).alias('no_of_4s_in_po'),
            sum(when(col('is_powerplay')==1,col('total_runs'))).alias('runs_conceeded_in_boundaries_po'),
            
            # 4s in Middle Overs
            count(when(col('is_middle_overs')==1,col('total_runs'))).alias('no_of_4s_in_mo'),
            sum(when(col('is_middle_overs')==1,col('total_runs'))).alias('runs_conceeded_in_boundaries_mo'),
            
            # 4s in Death Overs
            count(when(col('is_death_overs')==1,col('total_runs'))).alias('no_of_4s_in_do'),
            sum(when(col('is_death_overs')==1,col('total_runs'))).alias('runs_conceeded_in_boundaries_do') 
            )

In [0]:
df_no_of_6s = df_period.filter(col('total_runs')==6)\
                          .groupBy('bowler')\
                          .agg(
                              count(col('total_runs')).alias('no_of_6s'),
                              sum(col('total_runs')).alias('runs_conceeded_in_sixes'),

                              # 6s in PowerPlay
                              count(when(col('is_powerplay')==1,col('total_runs'))).alias('no_of_6s_in_po'),
                              sum(when(col('is_powerplay')==1,col('total_runs'))).alias('runs_conceeded_in_sixes_po'),
                              
                              # 6s in Middle Overs
                              count(when(col('is_middle_overs')==1,col('total_runs'))).alias('no_of_6s_in_mo'),
                              sum(when(col('is_middle_overs')==1,col('total_runs'))).alias('runs_conceeded_in_sixes_mo'),
                              
                              # 6s in Death Overs
                              count(when(col('is_death_overs')==1,col('total_runs'))).alias('no_of_6s_in_do'),
                              sum(when(col('is_death_overs')==1,col('total_runs'))).alias('runs_conceeded_in_sixes_do')
                          )

> #### Joins for Boundary and Sixes 

In [0]:
df_4s_6s_info = df_no_of_4s.join(df_no_of_6s, how='inner', on='bowler')

In [0]:
df_4s_6s_info.display()

#### Boundary Percentage Calculation

In [0]:
df_boundary_calc = df_4s_6s_info.join(df_total_overs, how='inner', on='bowler')\
            .groupBy('bowler')\
            .agg(
                # Total Boundary Percentage
                round(
                    (sum('runs_conceeded_in_boundaries')+sum('runs_conceeded_in_sixes'))/sum('total_runs_given')*100
                    )
                .alias('total_boundary_per'),

                # Boundary Percentage in PowerPlay
                round(
                    (sum('runs_conceeded_in_boundaries_po')+sum('runs_conceeded_in_sixes_po'))/sum('runs_in_po')*100
                    )
                .alias('boundary_per_in_po'),

                # Boundary Percentage in Middle Overs
                round(
                    (sum('runs_conceeded_in_boundaries_mo')+sum('runs_conceeded_in_sixes_mo'))/sum('runs_in_mo')*100
                    )
                .alias('boundary_per_in_mo'),

                # Boundary Percentage in Death Overs
                round(
                    (sum('runs_conceeded_in_boundaries_do')+sum('runs_conceeded_in_sixes_do'))/sum('runs_in_do')*100
                    )
                .alias('boundary_per_in_do')
                )\
                               .select(
                                   'bowler',
                                   'total_boundary_per',
                                   'boundary_per_in_po',
                                   'boundary_per_in_mo',
                                   'boundary_per_in_do'
                                   )

In [0]:
df_boundary_per = df_boundary_calc.join(df_4s_6s_info, how='inner', on='bowler')\
                            .select(
                                 'bowler',
                                 'no_of_4s',
                                 'runs_conceeded_in_boundaries',
                                 'no_of_4s_in_po',
                                 'runs_conceeded_in_boundaries_po',
                                 'no_of_4s_in_mo',
                                 'runs_conceeded_in_boundaries_mo',
                                 'no_of_4s_in_do',
                                 'runs_conceeded_in_boundaries_do',
                                 'no_of_6s',
                                 'runs_conceeded_in_sixes',
                                 'no_of_6s_in_po',
                                 'runs_conceeded_in_sixes_po',
                                 'no_of_6s_in_mo',
                                 'runs_conceeded_in_sixes_mo',
                                 'no_of_6s_in_do',
                                 'runs_conceeded_in_sixes_do',
                                 'total_boundary_per',
                                 'boundary_per_in_po',
                                 'boundary_per_in_mo',
                                 'boundary_per_in_do'
                                 )

In [0]:
df_boundary_per.sort("total_boundary_per", ascending = False).display()

#### Dot Ball Metrics

In [0]:
df_dotball_per = df_period.groupBy('bowler').agg(
                    count(when(col('total_runs')==0,1)).alias('no_of_dotballs'),
                    count(when(((col('wides')==0) & (col('noballs')==0)),col('over')))\
                        .alias('no_of_legal_balls'),

                    # DotBall in PowerPlay
                    count(when((col('total_runs')==0)&(col('is_powerplay')==1),1)).alias('no_of_dotballs_po'),
                    count(when(((col('wides')==0) & (col('noballs')==0))&(col('is_powerplay')==1),col('over')))\
                        .alias('no_of_legal_balls_po'),

                    # DotBall in Middle Overs
                    count(when((col('total_runs')==0)&(col('is_middle_overs')==1),1)).alias('no_of_dotballs_mo'),
                    count(when(((col('wides')==0) & (col('noballs')==0))&(col('is_middle_overs')==1),col('over')))\
                        .alias('no_of_legal_balls_mo'),

                    # DotBall in Death Overs
                    count(when((col('total_runs')==0)&(col('is_death_overs')==1),1)).alias('no_of_dotballs_do'),
                    count(when(((col('wides')==0) & (col('noballs')==0))&(col('is_death_overs')==1),col('over')))\
                        .alias('no_of_legal_balls_do')
                        )\
                    .withColumn('dotball_per',
                                round(
                                    (col('no_of_dotballs')/col('no_of_legal_balls'))*100)
                                )\
                    .withColumn('dotball_per_in_po',
                                round(
                                try_divide(col('no_of_dotballs_po'),col('no_of_legal_balls_po'))*100)
                                )\
                    .withColumn('dotball_per_in_mo',
                                round(
                                try_divide(col('no_of_dotballs_mo'),col('no_of_legal_balls_mo'))*100)
                                )\
                    .withColumn('dotball_per_in_do',
                                round(
                                try_divide(col('no_of_dotballs_do'),col('no_of_legal_balls_do'))*100)
                                )

In [0]:
df_dotball_final_select = df_dotball_per.select(
                'bowler',
                'no_of_legal_balls',
                'no_of_dotballs',
                'dotball_per',
                'no_of_legal_balls_po',
                'no_of_dotballs_po',
                'dotball_per_in_po',
                'no_of_legal_balls_mo',
                'no_of_dotballs_mo',
                'dotball_per_in_mo',
                'no_of_legal_balls_do',
                'no_of_dotballs_do',
                'dotball_per_in_do')

#### Balls per Boundary Calculation

In [0]:
df_ball_per_bound_join = df_boundary_per.join(df_dotball_per, on='bowler', how='inner')\
 .withColumn('balls_per_boundary', round(try_divide(col('no_of_legal_balls'),
    (col('no_of_4s')+col('no_of_6s'))),2))\
 .withColumn('balls_per_boundary_in_po', round(try_divide(col('no_of_legal_balls_po'),
    (col('no_of_4s_in_po')+col('no_of_6s_in_po'))),2))\
 .withColumn('balls_per_boundary_in_mo', round(try_divide(col('no_of_legal_balls_mo'),
    (col('no_of_4s_in_mo')+col('no_of_6s_in_mo'))),2))\
 .withColumn('balls_per_boundary_in_do', round(try_divide(col('no_of_legal_balls_do'),
    (col('no_of_4s_in_do')+col('no_of_6s_in_do'))),2))        

In [0]:
df_ball_per_bound_join.display()

#### Using window functions to get runs conceeded in each innings

In [0]:
innings_window = Window.partitionBy('bowler','match_id').orderBy(col('match_id').asc())

In [0]:
df_window = df_period.withColumn('total_runs_per_match', sum('total_runs').over(innings_window))

In [0]:
df_bowler_runs_each_innings = df_window\
    .select('bowler','match_id','total_runs_per_match').distinct()

In [0]:
df_runs_bucket = df_bowler_runs_each_innings.withColumn(
    "Runs_Bucket",
    when(col("total_runs_per_match") < 25, "0-25")
    .when(col("total_runs_per_match") < 40, "25-40")
    .otherwise("40+")
)

In [0]:
df_runs_bucket_pivoted = (
    df_runs_bucket
    .groupBy("bowler")
    .pivot("Runs_Bucket")
    .agg(count("Runs_Bucket").alias("Times"))
)

In [0]:
df_runs_bucket_pivoted.sort("0-25", ascending=False).display()

In [0]:
final_df = bowling_metrics_df.join(df_wides_noballs, on='bowler', how='left')\
                             .join(df_boundary_per, on='bowler', how='left')\
                             .join(df_dotball_final_select, on='bowler', how='left')\
                             .join(df_ball_per_bound_join, on='bowler', how='left')\
                             .join(df_runs_bucket_pivoted, on='bowler', how='left')